# Stage 01 — Data Quality Assessment

**Purpose:** Univariate analysis and data quality assessment on the German Credit dataset.

**Inputs:**
- `data/loans.csv` — raw dataset (1000 observations, 21 variables)
- Target: `Creditability` (1 = default)

In [ ]:
import os, sys
os.chdir(r'C:\projects\superagent')
sys.path.insert(0, 'src')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pdtoolkit as pdt

RUN_DIR = 'runs/2026-03-17_071354'

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

BLUE = '#2166AC'
RED = '#D6604D'
GREY = '#999999'

In [ ]:
# Load dataset
db = pd.read_csv('data/loans.csv')
print(f'Dataset shape: {db.shape}')
print(f'Target default rate: {db["Creditability"].mean():.4f} ({db["Creditability"].sum():.0f}/{len(db)})')

In [ ]:
# Load variable types reference
var_types = pd.read_csv('data/variable_types.csv')
print(var_types[['Variable', 'Type', 'dtype', 'Monotonicity', 'SpecialCodes']].to_string(index=False))

In [ ]:
# Univariate analysis
uv = pdt.univariate(db)
print(uv.to_string())

In [ ]:
# Near-zero variance detection
nzv = pdt.nzv(db)
print('Near-zero variance analysis:')
# Flag variables with frequency ratio > 19 (95:5 rule)
nzv_flagged = nzv[nzv['cc_fqr'].astype(float) > 19]
if len(nzv_flagged) > 0:
    print(f'Flagged variables (freq ratio > 19): {list(nzv_flagged["rf"])}')
else:
    print('No near-zero variance variables flagged.')
print()
print(nzv[['rf', 'cc_unv', 'cc_fqr']].to_string())

In [ ]:
# Missing value analysis
missing = db.isnull().sum()
missing_pct = (missing / len(db) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
print('Missing values per variable:')
print(missing_df.to_string())

# Plot missing rates
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(missing_df.index, missing_df['pct'], color=BLUE)
ax.set_xlabel('Missing Rate (%)')
ax.set_title('Missing Value Rates by Variable')
ax.invert_yaxis()
for i, v in enumerate(missing_df['pct']):
    ax.text(v + 0.1, i, f'{v:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/01_missing_rates.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# Distribution plots for all variables
features = [c for c in db.columns if c != 'Creditability']
n_vars = len(features)
n_cols = 4
n_rows = (n_vars + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    if db[col].nunique() <= 10:
        # Bar plot for discrete/categorical
        vc = db[col].value_counts().sort_index()
        ax.bar(vc.index.astype(str), vc.values, color=BLUE, alpha=0.7)
    else:
        # Histogram for continuous
        ax.hist(db[col].dropna(), bins=30, color=BLUE, alpha=0.7, edgecolor='white')
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Variable Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/01_distributions.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# Correlation matrix for numeric variables
numeric_cols = [c for c in db.columns if c != 'Creditability']
corr = db[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7},
            vmin=-1, vmax=1)
ax.set_title('Correlation Matrix (Numeric Variables)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/01_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.close()

# Identify high correlation pairs
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append((corr.columns[i], corr.columns[j], round(r, 4)))

print(f'High correlation pairs (|r| > 0.7): {len(high_corr)}')
for pair in high_corr:
    print(f'  {pair[0]} <-> {pair[1]}: r = {pair[2]}')

In [ ]:
# Outlier analysis for continuous variables using IQR method
continuous_vars = ['Duration of Credit (month)', 'Credit Amount', 'Age (years)']
print('Outlier Analysis (IQR method, 1.5x):')
print(f'{"Variable":<35} {"Q1":>8} {"Q3":>8} {"IQR":>8} {"Lower":>8} {"Upper":>8} {"N_out":>6} {"Pct":>6}')
for v in continuous_vars:
    q1 = db[v].quantile(0.25)
    q3 = db[v].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((db[v] < lower) | (db[v] > upper)).sum()
    pct_out = n_out / len(db) * 100
    print(f'{v:<35} {q1:>8.1f} {q3:>8.1f} {iqr:>8.1f} {lower:>8.1f} {upper:>8.1f} {n_out:>6d} {pct_out:>5.1f}%')
print()
print('Justification: IQR method selected as it is robust to skewed distributions.')
print('All three continuous variables show right-skewed distributions; IQR-based capping')
print('provides conservative upper bounds without excessive data alteration.')

In [ ]:
# Variable action summary
var_actions = {
    'Account Balance': 'keep (ordinal, special code 4=no checking account)',
    'Duration of Credit (month)': 'impute-outliers (IQR cap, 70 outliers / 7.0%)',
    'Payment Status of Previous Credit': 'keep (categorical)',
    'Purpose': 'keep (nominal/categorical)',
    'Credit Amount': 'impute-outliers (IQR cap, 72 outliers / 7.2%)',
    'Value Savings/Stocks': 'keep (ordinal, special code 5=no savings)',
    'Length of current employment': 'keep (ordinal)',
    'Instalment per cent': 'keep (numerical)',
    'Sex & Marital Status': 'keep (nominal/categorical)',
    'Guarantors': 'keep (nominal/categorical)',
    'Duration in Current address': 'keep (numerical)',
    'Most valuable available asset': 'keep (ordinal, special code 4=no property)',
    'Age (years)': 'impute-outliers (IQR cap, 23 outliers / 2.3%)',
    'Concurrent Credits': 'keep (nominal/categorical)',
    'Type of apartment': 'keep (nominal/categorical)',
    'No of Credits at this Bank': 'keep (numerical)',
    'Occupation': 'keep (ordinal)',
    'No of dependents': 'keep (numerical)',
    'Telephone': 'keep (binary/categorical)',
    'Foreign Worker': 'keep (binary/categorical, NZV flagged: freq ratio 26.0)'
}

print('Variable Action Summary:')
print(f'{"Variable":<40} {"Action"}')
print('-' * 80)
for var, action in var_actions.items():
    print(f'{var:<40} {action}')

## Stage Summary

| Item | Value | Status |
|---|---|---|
| Observations | 1000 | PASS |
| Variables (excl. target) | 20 | PASS |
| Default rate | 30.0% | PASS |
| Missing values | 0 across all variables | PASS |
| Near-zero variance | Foreign Worker (freq ratio 26.0) | WARN |
| High correlation pairs | 0 (none above |r| > 0.7) | PASS |
| Outlier imputation needed | 3 variables (Duration, Amount, Age) | PASS |
| Variables to exclude | 0 | PASS |

**Flags for human review:** Foreign Worker has high frequency ratio (26.0) suggesting near-zero variance — monitor discriminatory power in Stage 03.

**Recommended action for next stage:** Proceed to Stage 02 (Data Preparation). Cap outliers in Duration of Credit, Credit Amount, and Age using IQR method. No missing value imputation needed. No variables recommended for exclusion at this stage.